# Pauli string generator tester!
## base case model
we consider a grid with 5 rows and 4 columns of qubit positions, giving 20 qubits.

Each elementary cell contributes one independent generator:
- Even cell rows: XXX on the top-left, bottom-left, and bottom-right vertices.
- Odd cell rows: ZZZ on the top-left, top-right, and bottom-left vertices.

Each Pauli string P represents the skew-Hermitian generator iP. The program stores only the Pauli string labels.

The expected base-case dimension is **70**.

## extending the model
To enlarge the grid while keeping the same interaction pattern:
1. Change rows and cols, which count qubit positions.
2. Rerun the model cell and all subsequent code cells.
3. The program will rebuild the triples and generators automatically.
4. Remove or update the base-case check asserting dimension == 70.

This extension preserves the alternating X/Z rows and the L-shaped interactions.

For a different interaction pattern, modify x_triples and z_triples.
For arbitrary Pauli-string inputs, directly replace generators with a list of equal-length strings containing only I, X, Y, and Z.

In [1]:
# put initial generators here!
# generate triples

# setup: qubit positions
rows = 5
cols = 4
n_qubits = rows * cols

# qubit indices
# 0  1  2  3
# 4  5  6  7
# 8  9  10 11
# 12 13 14 15
# 16 17 18 19
def qubit_label(row, col):
    return row * cols + col

# initial generators
x_triples = []
z_triples = []

for i in range(rows - 1):
    for j in range(cols - 1):
        if i % 2 == 0:
            # top-left, bottom-left, bottom-right
            triple = (qubit_label(i, j), qubit_label(i + 1, j), qubit_label(i + 1, j + 1),)
            x_triples.append(triple)
        else:
            # top-left, top-right, bottom-left
            triple = (qubit_label(i, j), qubit_label(i, j + 1), qubit_label(i + 1, j),)
            z_triples.append(triple)

print("X triples:", x_triples)
print("Z triples:", z_triples)
print("Number of X triples:", len(x_triples))
print("Number of Z triples:", len(z_triples))

X triples: [(0, 4, 5), (1, 5, 6), (2, 6, 7), (8, 12, 13), (9, 13, 14), (10, 14, 15)]
Z triples: [(4, 5, 8), (5, 6, 9), (6, 7, 10), (12, 13, 16), (13, 14, 17), (14, 15, 18)]
Number of X triples: 6
Number of Z triples: 6


In [2]:
# make Pauli string
# convert the model's interaction supports into full Pauli strings.
def make_pauli_string(n_qubits, operator, sites):
    letters = ["I"] * n_qubits
    # replace at selected positions
    for site in sites:
        letters[site] = operator
    return "".join(letters)

print(make_pauli_string(n_qubits, "X", x_triples[0]),'\n')

# make generators
generators = []

for triple in x_triples:
    pauli = make_pauli_string(n_qubits, "X", triple)
    generators.append(pauli)
for triple in z_triples:
    pauli = make_pauli_string(n_qubits, "Z", triple)
    generators.append(pauli)

for i, pauli in enumerate(generators):
    print(i, pauli)
    
print('')
print("Number of generators:", len(generators))

XIIIXXIIIIIIIIIIIIII 

0 XIIIXXIIIIIIIIIIIIII
1 IXIIIXXIIIIIIIIIIIII
2 IIXIIIXXIIIIIIIIIIII
3 IIIIIIIIXIIIXXIIIIII
4 IIIIIIIIIXIIIXXIIIII
5 IIIIIIIIIIXIIIXXIIII
6 IIIIZZIIZIIIIIIIIIII
7 IIIIIZZIIZIIIIIIIIII
8 IIIIIIZZIIZIIIIIIIII
9 IIIIIIIIIIIIZZIIZIII
10 IIIIIIIIIIIIIZZIIZII
11 IIIIIIIIIIIIIIZZIIZI

Number of generators: 12


In [3]:
# single qubit Pauli multiplication table
# each entry stores (phase factor, resulting operator)
pauli_product = {
    ("I", "I"): (1, "I"),
    ("I", "X"): (1, "X"),
    ("I", "Y"): (1, "Y"),
    ("I", "Z"): (1, "Z"),

    ("X", "I"): (1, "X"),
    ("X", "X"): (1, "I"),
    ("X", "Y"): (1j, "Z"),
    ("X", "Z"): (-1j, "Y"),

    ("Y", "I"): (1, "Y"),
    ("Y", "X"): (-1j, "Z"),
    ("Y", "Y"): (1, "I"),
    ("Y", "Z"): (1j, "X"),

    ("Z", "I"): (1, "Z"),
    ("Z", "X"): (1j, "Y"),
    ("Z", "Y"): (-1j, "X"),
    ("Z", "Z"): (1, "I"),
}

# multiply string
def multiply_strings(p, q):
    if len(p) != len(q):
        raise ValueError("Pauli strings must have the same length.")

    # multiply corresponding single qubit operation
    coefficient = 1
    letters = []

    # tensor product multiplication
    for a, b in zip(p, q):
        local_coefficient, operator = pauli_product[(a, b)]

        coefficient *= local_coefficient
        letters.append(operator)

    return coefficient, "".join(letters)

# ex
print(multiply_strings("ZIIIII", "XXIXII"))
print(multiply_strings("XXIXII", "ZIIIII"))

(1j, 'YXIXII')
(-1j, 'YXIXII')


In [4]:
# compute commutator
# either 0 or 2pq
def commutator(p, q):
    if len(p) != len(q):
        raise ValueError("Pauli strings must have the same length.")

    # count positions where the single-qubit operators anticommute.
    mismatches = 0
    for a, b in zip(p, q):
        if a != "I" and b != "I" and a != b:
            mismatches += 1

    # even number of local anticommutations -> PQ = QP.
    if mismatches % 2 == 0:
        return 0, None

    # otherwise PQ = -QP -> [P, Q] = 2PQ.
    coefficient, result = multiply_strings(p, q)
    return 2 * coefficient, result


In [5]:
# generate lie algebra
def generate_lie_algebra(generators, verbose=False, max_basis=100000):
    # remove duplicate input generators.
    original = list(dict.fromkeys(generators))

    # validate input before starting the search.
    if not original:
        raise ValueError("provide at least one generator.")
    if any(not isinstance(p, str) for p in original):
        raise ValueError("each generator must be a string.")
    n = len(original[0])
    if n == 0 or any(len(p) != n for p in original):
        raise ValueError("strings must have the same positive length.")
    if any(set(p) - set("IXYZ") for p in original):
        raise ValueError("only I, X, Y, Z are allowed.")
    if max_basis < len(original):
        raise ValueError("max_basis is smaller than the input set.")

    basis = original.copy() # stores discovered directions in processing order
    known = set(basis) # provides fast membership checks
    index = 0 # identifies the next direction to process

    # newly discovered directions are appended and processed later
    # BFS
    while index < len(basis):
        p = basis[index]

        # original generators suffice to span all nested commutators.
        for q in original:
            coefficient, new_string = commutator(p, q)
            if coefficient != 0 and new_string not in known:
                if len(basis) >= max_basis:
                    raise RuntimeError(
                        f"Incomplete search: at least {len(basis)} "
                        "basis limit reached."
                    )

                # nonzero scalar multiples represent the same direction.
                known.add(new_string)
                basis.append(new_string)

                if verbose:
                    print(
                        f"[{p}, {q}] = "
                        f"{coefficient} * {new_string}"
                    )

        index += 1

    return basis

In [6]:
# compute the closure for the current model
basis = generate_lie_algebra(generators)

# count independent Pauli directions
dimension = len(basis)
print("Dimension:", dimension)

# check computed dimensions.
assert dimension == 70
# check process
# basis = generate_lie_algebra(generators, verbose=True)

Dimension: 70
